# 🎯 [K15_M09] 여러 조건을 합쳐 시설 추천하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m09_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 09. 여러 조건을 합쳐 시설 추천하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 09. 여러 조건을 합쳐 시설 추천하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>시간대, 수용 인원, 기자재(빔프로젝터 등) 구비 여부 등 다중 조건을 만족하는 최적의 이용 시설을 도출하는 복합 추천 프롬프트를 구현합니다. 시설 예약 현황뿐만 아니라 최근 점검 이력까지 교차 검토하여 기자재 잦은 고장 이력이 있는 시설은 주의 사항으로 사전 안내하도록 설계합니다. AI의 추천을 참고하되 최종 승인 및 결정은 담당자가 수행하는 원칙을 체감합니다.</p>

**[학습 목표 및 체득 역량]**
<p>시간, 인원, 장비 등 복합 조건을 조합하여 최적 시설 후보를 도출하고, 점검 이력과 교차 검토하여 장비 고장 주의 사항을 반영합니다. <strong>이를 통해 복수의 데이터셋을 교차 분석하여 정밀한 추천 결과를 도출하고, AI 제안에 대한 인간의 최종 의사결정 권한과 책임을 확립하는 역량을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 조건 3가지를 합친 질문을 만든다.
<h4>이 단계의 역할</h4><p>시간, 연속 사용 시간, 예상 인원, 필수 장비 등 복합 조건을 조합한 실무 시나리오 기반의 다중 조건 질의를 생성합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>복합 조건 시나리오를 구성합니다: '목요일', '2시간 연속 사용', '15명 내외 토의', '빔프로젝터 사용 필수'.</p></li>
<li><p>해당 조건을 명시한 종합 시설 추천 요청 질문을 작성합니다.</p></li>
<li><p>단순 가용 여부를 넘어 시설의 특성과 장비 지원 여부까지 질의에 포함합니다.</p></li>
</ol>
<h4>💡 다중 조건 질의 작성 예시</h4><blockquote><p><strong>[복합 조건 추천 질의]:</strong> "[예약 현황표 및 점검 이력표 포함]<br>[조건]:<br>- 요일: 목요일<br>- 시간: 14시 이후 2시간 연속 사용 희망<br>- 인원: 약 15명 분대장 토의<br>- 장비: 빔프로젝터를 이용한 교육 자료 투사 필요<br>위 조건을 만족하는 가장 적합한 시설 후보를 추천하고 근거와 주의사항을 제시해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>조건 구체성:</strong> 요일(목), 시간(14시 이후), 규모(15명), 장비(빔프로젝터)가 명확히 명시되었는가?</li>
<li><strong>선택지 존재성:</strong> 예약표 상 목요일 14시 이후 세미나실 A(14-18 빈 상태)와 세미나실 B(14-16 빈, 16-18 AI 스터디)가 후보군으로 추출 가능한가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>다중 조건 질의 시나리오 작성 완료</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>다중 조건 추천 질문문</p></li></ul>

#### 🔹 Step 2: 점검 이력을 함께 보고 후보를 고르게 한다.
<h4>이 단계의 역할</h4><p>빈 시간대 예약표뿐만 아니라 점검 이력표(설비 고장 내역)를 함께 교차 분석하여, 장비 신뢰도를 반영한 최적 후보를 선별합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>목요일 14-16시가 비어 있는 '세미나실 A'와 '세미나실 B'를 1차 후보로 도출합니다.</p></li>
<li><p>점검 이력표를 조회하여 세미나실 B가 8~10월 중 빔프로젝터 고장이 3회(H01, H06, H16) 발생했음을 확인합니다.</p></li>
<li><p>AI가 세미나실 A를 우선 추천하고, 세미나실 B를 추천할 경우 '빔프로젝터 잦은 고장 이력 주의'를 경고하도록 분석합니다.</p></li>
</ol>
<h4>💡 AI 교차 분석 추천 질의 예시</h4><blockquote><p><strong>[교차 검토 추천 실행]:</strong> "예약표 상 비어 있는 시설 후보들을 나열한 후, [점검 이력표]의 설비 고장 기록(특히 빔프로젝터)을 대조하여 장비 작동 안정성이 더 높은 시설을 1순위로 추천하고 그 이유를 설명해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>교차 검토 반영:</strong> 예약표의 '빈' 상태만 보지 않고 점검 이력표의 결함 데이터를 함께 반영했는가?</li>
<li><strong>우선순위 타당성:</strong> 세미나실 B의 빔프로젝터 위험을 명시하고 안정적인 대안을 제시했는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>예약 현황과 점검 이력을 교차 반영한 추천 결과 도출</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>시설 추천 결과 및 데이터 교차 대조표</p></li></ul>

#### 🔹 Step 3: 주의 사항과 사람 확인 단계를 적는다.
<h4>이 단계의 역할</h4><p>AI의 추천 결과에 '장비 사전 점검 권고' 등 실무 주의사항을 표기하고, 최종 승인 및 예약은 담당자가 수행한다는 책임 한계를 정립합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>추천 보고서에 '세미나실 B 이용 시 행사 30분 전 빔프로젝터 정상 투사 여부 필수 점검' 주의사항을 명기합니다.</p></li>
<li><p>'본 추천은 AI의 데이터 분석 결과일 뿐이며, 공식 승인 및 예약 신청은 대대 행정과(내선 1101)를 통해 지휘관 결재 후 확정된다'는 인간 책임 조항을 작성합니다.</p></li>
<li><p>최종 추천 요약 카드 3건을 완성합니다.</p></li>
</ol>
<h4>💡 AI 책임 한계 명시 예시</h4><blockquote><p><strong>[주의사항 및 권한 명시]:</strong> "추천 결과 하단에 (1) 이용 전 실무자가 현장에서 확인해야 할 장비 주의사항과 (2) AI는 의사결정 보조 도구이며 최종 승인은 부서 담당자가 한다는 면책/권한 안내 문구를 작성해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>주의사항 구체성:</strong> 빔프로젝터 사전 점검 등 현장 작동 확인 조치가 포함되었는가?</li>
<li><strong>인간 책임 원칙(HITL):</strong> AI가 최종 결정을 내리는 것이 아님을 명시했는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>주의사항 및 인간 책임 권한이 명시된 최종 추천서 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>추천 결과 보고서 (주의사항 및 안내 문구 포함)</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>조건 중 하나(예: 인원을 15명에서 50명으로 변경, 또는 장비를 음향 방송 장비로 변경)를 수정하여 추천 결과가 어떻게 유동적으로 재산출되는지 분석합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>조건을 '인원 50명 강당 교육' 또는 '음향 마이크 필수 사용'으로 변경하여 재질의합니다.</p></li>
<li><p>AI가 세미나실 대신 '대대 강당'을 도출하는지, 강당의 무선 마이크 고장 이력(H05, H13, H21 미완료)을 감지하여 경고하는지 확인합니다.</p></li>
<li><p>조건 변화에 따른 추천 알고리즘의 민감도와 데이터 정합성을 평가합니다.</p></li>
</ol>
<h4>💡 AI 조건 변형 질의 예시</h4><blockquote><p><strong>[조건 변경 시뮬레이션]:</strong> "인원 조건을 50명으로 늘리고 '무선 마이크 2대 이상 사용'으로 조건을 변경하면 추천 시설이 어디로 바뀌며, 점검 이력 상 어떤 주의사항이 새롭게 추가되는지 분석해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>시설 전환 타당성:</strong> 대규모 인원에 맞춰 대대 강당이 후보로 변경되었는가?</li>
<li><strong>새로운 결함 감지:</strong> 강당 추천 시 마이크 수신 불량 접수(H21) 상태를 주의사항으로 정확히 짚어냈는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>조건 변경에 따른 추천 시설 및 주의사항 재산출 검증</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 조건 변경 추천 분석 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 다중 조건 시설 추천 결과 보고서
- **의뢰 조건**: 목요일 14시 이후 2시간 연속, 15명 토의, 빔프로젝터 사용
- **1차 후보 도출**: 세미나실 A (14-16 빈), 세미나실 B (14-16 빈)
- **점검 이력 교차 검토**: 세미나실 B는 8~10월 중 빔프로젝터 고장 3회(H01, H06, H16) 발생. 세미나실 A는 영상 장비 고장 이력 없음.

## 2. 최종 추천 요약 카드
```
[시설 추천 결과]
1순위 추천: 세미나실 A
- 가용 시간: 목요일 14:00 ~ 16:00 (16-18시도 비어 있어 연장 가능)
- 추천 사유: 15명 토의에 적합하며, 최근 설비 고장 이력 없이 안정적임.
- 주의 사항: 특이사항 없음.

2순위 추천: 세미나실 B
- 가용 시간: 목요일 14:00 ~ 16:00 (16시 이후 AI 스터디 예약됨)
- 주의 사항: 최근 빔프로젝터 화면 깜빡임 등 고장 이력 3회 있음. 이용 전 30분 전 정상 투사 여부 확인 필수.

[📢 장병 필독 확인 권한]
※ 본 추천은 AI의 일정 및 이력 분석에 기반한 참고 자료입니다.
※ 실제 시설 이용 최종 승인 및 예약 확정은 대대 행정과(내선 1101)를 거쳐야 합니다.
```

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 다중 조건 추천·인간 책임 — 예약 현황과 점검 이력을 교차 분석하여 최적 시설을 추천하고 인간 최종 승인 권한을 확립했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>다중 조건 시나리오(요일, 시간, 인원, 장비) 기반 시설 추천 도출</p></li><li><p>예약 현황과 점검 이력 교차 분석을 통한 장비 결함 주의사항 표기</p></li><li><p>인간 검증 및 행정과 최종 승인 권한(HITL 책임) 명시 완료</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>AI의 단일 데이터 검색을 넘어 복수 데이터를 교차 분석했는지, AI의 추천을 넘어 인간 실무자의 최종 의사결정 권한을 명확히 고지했는지 평가한다.</p></blockquote>


In [ ]:
# [미션 09] 프롬프트 및 실습 결과 기록
mission_09_prompt = """# [미션 09] 복합 조건 시설 추천 및 점검 이력 교차 분석 프롬프트
너는 부대 행사 지원 컨설턴트다.
예약 현황표와 점검 이력표를 교차 분석하여 다음 다중 조건에 가장 부합하는 시설을 추천하라:

[사용자 요청 조건]
- 이용 요일 및 시간: 목요일 14:00 이후 2시간 연속 사용 희망
- 인원 및 성격: 15명 내외 분대장 전술 토의
- 필수 장비: 빔프로젝터 사용 필수

[지시 사항]
1. 1차 가용 후보 시설을 모두 도출하라.
2. 최근 3개월 점검 이력표를 대조하여 빔프로젝터 고장 이력을 교차 분석하라.
3. 최종 1순위 추천 시설을 선정하고, 차순위 시설의 주의 사항을 명시하라.
4. AI 추천은 참고용이며 최종 승인은 부서 권한임을 밝히는 안내 문구를 포함하라.
"""

mission_09_result = """### 📋 다중 조건 시설 추천 및 교차 분석 보고서

#### 1. 복합 조건 분석 및 시설 평가
- **요청 조건**: 목요일 14시 이후 2시간 연속, 15명 토의, 빔프로젝터 필수
- **1차 후보 도출**:
  - 세미나실 A: 목요일 14-16(빈), 16-18(빈) -> 4시간 연속 가용
  - 세미나실 B: 목요일 14-16(빈) -> 2시간 가용 (16시 이후 AI스터디)
- **점검 이력 교차 검토**:
  - 세미나실 B: 8~10월 빔프로젝터 고장 3회(H01, H06, H16) 발생하여 장비 신뢰도 취약.
  - 세미나실 A: 영상 장비 고장 이력 전무, 안정적 구동 가능.

#### 2. 최종 추천 요약 카드
```
[시설 추천 결과]
■ 1순위 추천: 세미나실 A
  - 가용 시간: 목요일 14:00 ~ 16:00 (필요 시 16-18시까지 연장 가능)
  - 추천 사유: 연속 시간 확보 가능 및 빔프로젝터 고장 이력 없음 (안정적 진행 보장)
  - 시설 비품: 빔프로젝터 구비 완료

■ 2순위 안내 (대안): 세미나실 B
  - 가용 시간: 목요일 14:00 ~ 16:00 (16시 정시 반납 필수)
  - 주의 사항: 최근 빔프로젝터 화면 깜빡임 고장 3회 발생 이력 있음. 사용 전 군수과 확인 권장.

[📢 장병 필독 확인 권한]
※ 본 추천은 AI의 일정 및 이력 교차 분석에 기반한 보조 자료입니다.
※ 실제 시설 이용 최종 승인 및 예약 확정은 대대 행정과(내선 1101)를 거쳐야 합니다.
```
"""

print(f"미션 09 실습 기록 완료: 프롬프트 {len(mission_09_prompt.strip())}자, 결과 {len(mission_09_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션9] 다중 조건(목요일 14시 이후, 빔프로젝터) 시설 필터링 및 점검 이력 조인
thu_slots = df_schedule[df_schedule['day'] == '목'][['facility', '14-16', '16-18']]
print("=== 목요일 오후(14시 이후) 시설 예약 현황 ===")
print(thu_slots)

proj_issues = df_maintenance[df_maintenance['content'].str.contains("빔프로젝터")]
print("\n=== 빔프로젝터 고장 이력 (세미나실 B 3건 확인) ===")
print(proj_issues[['id', 'date', 'facility', 'content', 'status']])
